In [86]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, NumeralTickFormatter, LinearAxis, Range1d, HoverTool
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10 # For distinct colors
from bokeh.io import output_notebook
from bokeh.layouts import row 

In [2]:
input_path = "../00_data/df_Popltn_SurgWt_combined.parquet"
df_for_eda = pd.read_parquet(input_path)
df_for_eda.head()


,HEALTH_AUTHORITY,HOSPITAL_NAME,PROCEDURE_GROUP,COMPLETED_50TH_PERCENTILE,COMPLETED_90TH_PERCENTILE,Calendar_Year,City,WAITING_INT,COMPLETED_INT,Population
0,Fraser,Abbotsford Regional Hospital And Cancer Centre,Abdominoplasty,0.0,0.0,2010,Abbotsford,10,5,135168.0
1,Fraser,Abbotsford Regional Hospital And Cancer Centre,All Other Procedures,4.0,34.2,2010,Abbotsford,30,59,135168.0
2,Fraser,Abbotsford Regional Hospital And Cancer Centre,Aortic Aneurysm Repair,5.1,12.1,2010,Abbotsford,18,14,135168.0
3,Fraser,Abbotsford Regional Hospital And Cancer Centre,Appendectomy,0.0,0.0,2010,Abbotsford,5,5,135168.0
4,Fraser,Abbotsford Regional Hospital And Cancer Centre,Biopsy in OR,1.9,8.8,2010,Abbotsford,6,37,135168.0


In [18]:
df_yearly_pop = df_for_eda.groupby('Calendar_Year')['Population'].mean().reset_index()
df_yearly_pop.rename(columns={'Population': 'Total_Population'}, inplace=True)
df_yearly_pop['Calendar_Year'] = df_yearly_pop['Calendar_Year'].astype(int)

output_notebook() # Enable Bokeh output in the notebook
# Create a ColumnDataSource object from the prepared DataFrame
source = ColumnDataSource(df_yearly_pop)

Loading BokehJS ...

In [40]:
tooltips_p = [
    ("Year", "@Calendar_Year"),
    ("Population", "@Total_Population{0,0}")]
# 1. Create the figure
p = figure(
    title="Population Growth Across All Cities (2010-2004)",
    x_axis_label="Calendar Year",
    y_axis_label="Total Population",
    tooltips=tooltips_p,
    height=350,
    width=600,
    tools="pan,wheel_zoom,box_zoom,reset,save" # Common tools
)

# 2. Add a line glyph
p.line(
    x='Calendar_Year',
    y='Total_Population',
    source=source,
    line_width=3,
    line_color="#1f77b4", # A nice blue color
    legend_label="Aggregated Population"
)

# 3. Add circles for each data point
p.circle(
    x='Calendar_Year',
    y='Total_Population',
    source=source,
    size=8,
    color="#1f77b4",
    fill_color="white",
    line_width=2
)

# 4. Format the Y-axis to display numbers with commas (e.g., 1,000,000)
p.yaxis.formatter = NumeralTickFormatter(format="0,0")

# 5. Ensure X-axis shows years as integers
p.xaxis.ticker = df_yearly_pop['Calendar_Year'].unique()

# Display the plot
#show(p)

In [41]:
df_yearly_wait = df_for_eda.groupby('Calendar_Year')['WAITING_INT'].mean().reset_index()
df_yearly_wait.rename(columns={'WAITING_INT': 'Avg_Wait_Time'}, inplace=True)
df_yearly_wait['Calendar_Year'] = df_yearly_wait['Calendar_Year'].astype(str) # Bokeh uses strings for categorical X-axis
source_wait = ColumnDataSource(df_yearly_wait)
x_axis_years = df_yearly_wait['Calendar_Year'].unique().tolist()
wait_max = df_yearly_wait['Avg_Wait_Time'].max() * 1.05
wait_min = df_yearly_wait['Avg_Wait_Time'].min() * 0.95

tooltips_p2 = [
    ("Year", "@Calendar_Year"),
    ("Avg Wait Time", "@Avg_Wait_Time{0.00} Days")
]
# 1. Create the figure
p2 = figure(
    title="Overall Average Surgical Wait Times in BC by Year",
    x_axis_label="Calendar Year",
    y_axis_label="Average Wait Time (Days)",
    x_range=x_axis_years, # Use string years for the categorical axis
    y_range=(wait_min, wait_max),
    tooltips=tooltips_p2,
    height=350,
    width=600,
    tools="pan,wheel_zoom,box_zoom,reset,save" # Common tools
)

# 2. Add a line glyph
p2.line(
    x='Calendar_Year',
    y='Avg_Wait_Time',
    source=source_wait,
    line_width=3,
    line_color="#d62728", # Red as wait color
    legend_label="Avg Wait Time"

)

# 3. Add circles for each data point
p2.circle(
    x='Calendar_Year',
    y='Avg_Wait_Time',
    source=source_wait,
    size=8,
    color="#d62728",
    fill_color="white",
    line_width=2
)

# 4. Format the Y-axis 
p2.yaxis.formatter = NumeralTickFormatter(format="0") # Simple integer format
p2.grid.grid_line_alpha = 0.3

# Display the plot
#show(p2)

In [42]:
p2.legend.location = "bottom_right"
p.legend.location = "bottom_right"

combined_layout = row(p, p2)
show(combined_layout)

In [115]:
# --- 1. DATA AGGREGATION & GROWTH RATE CALCULATION ---
# Calculate yearly average population and average wait time
df_yearly = df_for_eda.groupby('Calendar_Year').agg(
    Avg_Population=('Population', 'mean'),
    Avg_Wait_Time=('WAITING_INT', 'mean')
).reset_index()

# Sort by year to ensure correct growth calculation
df_yearly.sort_values(by='Calendar_Year', inplace=True)

# Calculate Year-over-Year Growth Rates (as a percentage)
df_yearly['Pop_Growth_Rate'] = df_yearly['Avg_Population'].pct_change() * 100
df_yearly['Wait_Time_Growth_Rate'] = df_yearly['Avg_Wait_Time'].pct_change() * 100

# Drop the first year (which has NaN for growth rate)
df_growth = df_yearly.dropna(subset=['Pop_Growth_Rate']).copy()
df_growth['Year'] = df_growth['Calendar_Year'].astype(str) # Convert year to string for categorical X-axis

# Prepare data for Bokeh
source = ColumnDataSource(df_growth)
years_sorted = df_growth['Year'].tolist()

# Define consistent colors
POP_COLOR = '#1f77b4'  # Blue
WAIT_COLOR = '#d62728' # Red


# --- 2. CREATE DUAL AXIS BOKEH CHART ---

# Setup the Primary Figure (Y-axis 1: Population Growth Rate)
p_dual_axis = figure(
    x_range=years_sorted, 
    height=450,
    width=800,
    title="Comparison of Annual Population and Wait Time Growth Rates",
    x_axis_label="Year-over-Year Change (Start Year)",
    y_axis_label="Population Growth Rate (%)",
    # Set the range for the primary (left) axis
    y_range=Range1d(start=df_growth[['Pop_Growth_Rate', 'Wait_Time_Growth_Rate']].min().min() * 1.1, 
                    end=df_growth[['Pop_Growth_Rate', 'Wait_Time_Growth_Rate']].max().max() * 1.1),
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

# Add the Primary Y-axis Series (Population Growth - Blue Line)
p_dual_axis.line(
    x='Year', 
    y='Pop_Growth_Rate', 
    source=source, 
    line_color=POP_COLOR, 
    line_width=3, 
    legend_label="Population Growth Rate"
)
p_dual_axis.circle(x='Year', y='Pop_Growth_Rate', source=source, size=8, color=POP_COLOR)
p_dual_axis.yaxis.formatter = NumeralTickFormatter(format="0.00%") # Format as percentage

# Create the Secondary Y-axis (Y-axis 2: Wait Time Growth Rate)
# We use the same Y-range name, but use a separate axis for layout control
p_dual_axis.extra_y_ranges = {
    "Wait_Range": p.y_range
}

# Add the Secondary Y-axis Layout
p_dual_axis.add_layout(
    LinearAxis(
        y_range_name="Wait_Range", 
        axis_label="Wait Time Growth Rate (%)", 
        formatter=NumeralTickFormatter(format="0.00%"),
        axis_line_color=WAIT_COLOR,
        major_label_text_color=WAIT_COLOR
    ), 
    'right'
)

# Add the Secondary Y-axis Series (Wait Time Growth - Red Line)
p_dual_axis.line(
    x='Year', 
    y='Wait_Time_Growth_Rate', 
    source=source, 
    line_color=WAIT_COLOR, 
    line_width=3, 
    y_range_name="Wait_Range", 
    legend_label="Wait Time Growth Rate"
)
p_dual_axis.circle(x='Year', y='Wait_Time_Growth_Rate', source=source, size=8, color=WAIT_COLOR, y_range_name="Wait_Range")


# Finalize appearance and add hover tool
p_dual_axis.xgrid.grid_line_color = None

# Custom Hover Tool to show both metrics
dual_axis_hover = HoverTool(tooltips=[
    ("Year", "@Year"),
    ("Pop. Growth", "@Pop_Growth_Rate{0.2f}%"),
    ("Wait Time Growth", "@Wait_Time_Growth_Rate{0.2f}%"),
])
p_dual_axis.add_tools(dual_axis_hover)

p_dual_axis.legend.location = "bottom_left"
p_dual_axis.legend.orientation = "horizontal"

show(p_dual_axis)

Top 3 Health Authorities by total waiting time: ['Fraser', 'Vancouver Coastal', 'Vancouver Island']



In [49]:
# Calculate the mean population for each City and Year
df_pop_min_max = df_for_eda.groupby(['City', 'Calendar_Year'])['Population'].mean().reset_index()

# Pivot the population data to easily isolate the start and end year populations
df_pop_pivot = df_pop_min_max.pivot(index='City', columns='Calendar_Year', values='Population')
df_pop_pivot.columns = df_pop_pivot.columns.astype(str)

# Find the earliest and latest years that have non-NaN population data
year_cols = df_pop_pivot.columns
first_year = year_cols[df_pop_pivot[year_cols].notna().any()].min()
last_year = year_cols[df_pop_pivot[year_cols].notna().any()].max()
year_diff = int(last_year) - int(first_year)

# Filter for the start and end year population data
df_pop_start = df_pop_min_max[df_pop_min_max['Calendar_Year'] == int(first_year)][['City', 'Population']].set_index('City').rename(columns={'Population': 'Pop_Start'})
df_pop_end = df_pop_min_max[df_pop_min_max['Calendar_Year'] == int(last_year)][['City', 'Population']].set_index('City').rename(columns={'Population': 'Pop_End'})

# Calculate aggregates for the bubble plot
df_aggregates = df_for_eda.groupby('City').agg(
    Avg_Wait_Time=('WAITING_INT', 'mean'),
    Total_Surgeries=('COMPLETED_INT', 'sum')
).reset_index()

# Merge all calculated data
df_plot_data = df_aggregates.merge(df_pop_start, on='City', how='inner').merge(df_pop_end, on='City', how='inner')

# Calculate Annual Population Growth Rate (percentage)
df_plot_data['Growth_Rate'] = (
    (df_plot_data['Pop_End'] / df_plot_data['Pop_Start'])**(1/year_diff) - 1
) * 100

# Scale total surgeries for bubble size (Bokeh uses pixels, scale it to a range like 10 to 60)
min_surgeries = df_plot_data['Total_Surgeries'].min()
max_surgeries = df_plot_data['Total_Surgeries'].max()
size_range_min = 10
size_range_max = 60

df_plot_data['Bubble_Size'] = (
    (df_plot_data['Total_Surgeries'] - min_surgeries) / (max_surgeries - min_surgeries)
) * (size_range_max - size_range_min) + size_range_min


# ---------------------------
# 2. BOKEH PLOTTING
# ---------------------------

# output_notebook() # Uncomment if running in a Jupyter environment

source = ColumnDataSource(df_plot_data)

# Define tooltips for interactivity
tooltips_scatter = [
    ("City", "@City"),
    ("Pop. Growth Rate", "@Growth_Rate{0.00}%"),
    ("Avg Wait Time", "@Avg_Wait_Time{0.0} Days"),
    ("Total Surgeries", "@Total_Surgeries{0,0}")
]

# Create the figure
p_scatter = figure(
    title=f"Avg Wait Time vs. Annual Population Growth Rate by City ({first_year}-{last_year})",
    x_axis_label=f"Annual Population Growth Rate ({first_year}-{last_year}, %)",
    y_axis_label="Average Wait Time (Days)",
    height=500,
    width=800,
    tools="pan,wheel_zoom,box_zoom,reset,save,hover",
    tooltips=tooltips_scatter
)

# Add the scatter plot with bubble size
p_scatter.scatter(
    x='Growth_Rate',
    y='Avg_Wait_Time',
    source=source,
    size='Bubble_Size',
    line_color="#1f77b4",
    fill_color="#1f77b4",
    fill_alpha=0.6,
    line_width=1
)

# Format axes
p_scatter.xaxis.formatter = NumeralTickFormatter(format="0.00")
p_scatter.yaxis.formatter = NumeralTickFormatter(format="0")
p_scatter.grid.grid_line_alpha = 0.3

# Add labels for City names (Bokeh uses text glyphs for annotation)
p_scatter.text(
    x='Growth_Rate', 
    y='Avg_Wait_Time', 
    text='City', 
    source=source,
    text_font_size="8pt", 
    text_color="#333333",
    text_align="center",
    text_baseline="middle",
    x_offset=0,
    y_offset=15 # Offset above the circle
)


# Save the plot (will save an interactive HTML file)
# p.save('bokeh_bubble_scatter_plot.html')
show(p_scatter)

In [117]:
# Find the Top 3 HEALTH_AUTHORITY by total WAITING_INT
top_3_authorities = (
    df_for_eda
    .groupby('HEALTH_AUTHORITY')['WAITING_INT']
    .sum()
    .nlargest(3)
    .index
    .tolist()
)

print(f"Top 3 Health Authorities by total waiting time: {top_3_authorities}\n")

Top 3 Health Authorities by total waiting time: ['Fraser', 'Vancouver Coastal', 'Vancouver Island']



In [72]:
# Group by Health Authority and calculate the median of WAITING_INT
df_median_wait = df_for_eda.groupby('HEALTH_AUTHORITY')['WAITING_INT'].median().reset_index()
df_median_wait.rename(columns={'WAITING_INT': 'Median_Wait_Time'}, inplace=True)

# Sort the data by Median Wait Time for a clean bar chart
df_median_wait.sort_values(by='Median_Wait_Time', ascending=False, inplace=True)

# Prepare data for Bokeh
source_authority_bar = ColumnDataSource(df_median_wait)
authorities_sorted = df_median_wait['HEALTH_AUTHORITY'].tolist()
num_authorities = len(authorities_sorted)

In [73]:
OFFICIAL_COLOR_MAP = {
    'Vancouver Island': '#1f77b4', # Blue (Standard Matplotlib Blue)
    'Island': '#1f77b4',           # Include common abbreviation if needed
    'Interior': '#2ca02c',         # Green
    'Northern': '#ff7f0e',         # Orange
    'Fraser': '#800000',           # Maroon
    'Vancouver Coastal': '#a6cee3',# Light Blue
    'VCH': '#a6cee3',              # Include common abbreviation if needed
    'Provincial Health Services Authority': '#8b4513', # Brown
    'PHSA': '#8b4515'              # Include common abbreviation if needed
}
def get_authority_color(authority_name):
    # 1. Exact Match Check
    if authority_name in OFFICIAL_COLOR_MAP:
        return OFFICIAL_COLOR_MAP[authority_name]
    
    # 2. Robust Partial/Substring Match Check (Handles "Vancouver Coastal" vs "VCH")
    for name, color in OFFICIAL_COLOR_MAP.items():
        if name in authority_name or authority_name in name:
            return color
            
    # 3. Fallback Color (if a new Authority appears)
    return '#aaaaaa' # Gray fallback

# Create the final list of colors in the sorted order of the DataFrame
custom_colors = [get_authority_color(auth) for auth in authorities_sorted]

In [74]:
# Define the color mapper (ensure the palette size is large enough)
palette_size = max(3, num_authorities)
color_map = factor_cmap('HEALTH_AUTHORITY', palette=Category10[palette_size], factors=authorities_sorted)

# Create the figure
p_auth_bar = figure(
    # Use the sorted list for the categorical X-axis range
    x_range=authorities_sorted, 
    height=450,
    width=750,
    title="Median Surgical Wait Time by Health Authority",
    x_axis_label="Health Authority",
    y_axis_label="Median Wait Time (Days)",
    # Include hover tool for interactivity
    tools="pan,wheel_zoom,box_zoom,reset,save,hover", 
    tooltips=[
        ("Authority", "@HEALTH_AUTHORITY"),
        ("Median Wait", "@Median_Wait_Time{0.0} Days") # Format to one decimal place
    ]
)

# Add the vertical bar glyphs
p_auth_bar.vbar(x='HEALTH_AUTHORITY', top='Median_Wait_Time', width=0.8, source=source_authority_bar, 
       line_color='black', 
       #fill_color=color_map)
       fill_color=factor_cmap('HEALTH_AUTHORITY', palette=custom_colors, factors=authorities_sorted))

# Customize appearance
p_auth_bar.xgrid.grid_line_color = None # Hide vertical grid lines
p_auth_bar.y_range.start = 0 # Ensure y-axis starts at zero
p_auth_bar.xaxis.major_label_orientation = 0.8 # Angle the labels slightly if names are long
p_auth_bar.yaxis.formatter = NumeralTickFormatter(format="0.0")

# Display the plot
show(p_auth_bar)

In [75]:
#add a Commented line here
#The above Viz shows the median wait times across different Health Authorities in BC 
#But the Fraser health authority has the highest median wait time which is expected due to its large population
#So, We are creating a Standardizing the wait times based on population of each Health Authority to get a better comparison

In [76]:
# 1.1 Calculate Average Wait Time
df_wait_avg = df_for_eda.groupby('HEALTH_AUTHORITY')['WAITING_INT'].mean().reset_index()
df_wait_avg.rename(columns={'WAITING_INT': 'Avg_Wait_Time'}, inplace=True)

# 1.2 Calculate Average Population (handles redundancy across records)
# Use mean() on Population which assumes the population is roughly constant across records for a given authority.
df_pop_avg = df_for_eda.groupby('HEALTH_AUTHORITY')['Population'].mean().reset_index()
df_pop_avg.rename(columns={'Population': 'Avg_Population'}, inplace=True)

# 1.3 Merge the two aggregates
df_combined_pop_and_wait = pd.merge(df_wait_avg, df_pop_avg, on='HEALTH_AUTHORITY')
# 1.4 Calculate the new metric: Avg Wait Time * (100,000 / Avg Population)
df_combined_pop_and_wait['Wait_Time_Per_100k'] = (
    df_combined_pop_and_wait['Avg_Wait_Time'] / (df_combined_pop_and_wait['Avg_Population'] / 100000)
)

# 1.5 Sort the data by the new metric (descending)
df_combined_pop_and_wait.sort_values(by='Wait_Time_Per_100k', ascending=False, inplace=True)

source_pop_wait_authWise = ColumnDataSource(df_combined_pop_and_wait)
authorities_sorted = df_combined_pop_and_wait['HEALTH_AUTHORITY'].tolist()
num_authorities = len(authorities_sorted)


In [77]:
OFFICIAL_COLOR_MAP = {
    'Vancouver Island': '#1f77b4', # Blue (Standard Matplotlib Blue)
    'Island': '#1f77b4',           # Include common abbreviation if needed
    'Interior': '#2ca02c',         # Green
    'Northern': '#ff7f0e',         # Orange
    'Fraser': '#800000',           # Maroon
    'Vancouver Coastal': '#a6cee3',# Light Blue
    'VCH': '#a6cee3',              # Include common abbreviation if needed
    'Provincial Health Services Authority': '#8b4513', # Brown
    'PHSA': '#8b4515'              # Include common abbreviation if needed
}
def get_authority_color(authority_name):
    # 1. Exact Match Check
    if authority_name in OFFICIAL_COLOR_MAP:
        return OFFICIAL_COLOR_MAP[authority_name]
    
    # 2. Robust Partial/Substring Match Check (Handles "Vancouver Coastal" vs "VCH")
    for name, color in OFFICIAL_COLOR_MAP.items():
        if name in authority_name or authority_name in name:
            return color
            
    # 3. Fallback Color (if a new Authority appears)
    return '#aaaaaa' # Gray fallback

# Create the final list of colors in the sorted order of the DataFrame
custom_colors = [get_authority_color(auth) for auth in authorities_sorted]

In [78]:
# Define the color mapper
palette_size = max(3, num_authorities)
color_map = factor_cmap('HEALTH_AUTHORITY', palette=Category10[palette_size], factors=authorities_sorted)

# Create the figure
p_bar_pop_wait_authWise = figure(
    # Use the sorted list for the categorical X-axis range
    x_range=authorities_sorted, 
    height=450,
    width=750,
    title="Average Wait Time Standardized by Population Served (Per 100k)",
    x_axis_label="Health Authority",
    y_axis_label="Avg Wait Time Per 100k Population (Days)",
    # Include hover tool for interactivity
    tools="pan,wheel_zoom,box_zoom,reset,save,hover", 
    tooltips=[
        ("Authority", "@HEALTH_AUTHORITY"),
        ("Wait Time per 100k", "@Wait_Time_Per_100k{0.2f} Days"),
        ("Actual Avg Wait", "@Avg_Wait_Time{0.1f} Days"),
        ("Avg Population", "@Avg_Population{0,0}")
    ]
)

# Add the vertical bar glyphs
p_bar_pop_wait_authWise.vbar(x='HEALTH_AUTHORITY', top='Wait_Time_Per_100k', width=0.8, source=source_pop_wait_authWise, 
       line_color='black', 
       #fill_color=color_map)
       fill_color=factor_cmap('HEALTH_AUTHORITY', palette=custom_colors, factors=authorities_sorted))

# Customize appearance
p_bar_pop_wait_authWise.xgrid.grid_line_color = None # Hide vertical grid lines
p_bar_pop_wait_authWise.y_range.start = 0 # Ensure y-axis starts at zero
p_bar_pop_wait_authWise.xaxis.major_label_orientation = 0.8 # Angle the labels slightly
p_bar_pop_wait_authWise.yaxis.formatter = NumeralTickFormatter(format="0.00") # Show two decimal places

# Display the plot
show(p_bar_pop_wait_authWise)

In [67]:
print(authorities_sorted)

['Vancouver Island', 'Interior', 'Northern', 'Fraser', 'Vancouver Coastal', 'Provincial Health Services Authority']


In [79]:

# Calculate the overall average wait time per city (for sorting)
df_city_avg = df_for_eda.groupby('City')['WAITING_INT'].mean()

# Get the list of the top 5 cities by highest overall average wait time
top_5_cities = df_city_avg.nlargest(5).index.tolist()

# Define the order for the Y-axis (highest overall wait time first)
city_order = top_5_cities

# Filter and aggregate the data (Avg Wait Time by City & Procedure)
df_plot = df_for_eda[df_for_eda['City'].isin(top_5_cities)]
df_plot_aggregated = df_plot.groupby(['City', 'PROCEDURE_GROUP'])['WAITING_INT'].mean().reset_index()
df_plot_aggregated.rename(columns={'WAITING_INT': 'Avg_Wait_Time'}, inplace=True)



In [81]:
df_pivot = df_plot_aggregated.pivot(
    index='City', 
    columns='PROCEDURE_GROUP', 
    values='Avg_Wait_Time'
).fillna(0)

# Calculate the sum of wait times per city (the denominator for normalization)
df_pivot['Total_Wait_Time'] = df_pivot.sum(axis=1)

# Calculate the proportional breakdown for the 100% stack
procedure_types = df_pivot.columns.drop('Total_Wait_Time').tolist()
df_proportions = df_pivot[procedure_types].div(df_pivot['Total_Wait_Time'], axis=0)
df_proportions['City'] = df_proportions.index
df_proportions['Total_Wait_Time'] = df_pivot['Total_Wait_Time'] # Keep the total for tooltips

# Convert to ColumnDataSource, ensuring the city order is maintained
df_proportions = df_proportions.set_index('City').loc[city_order].reset_index()
source_proc_top_cities = ColumnDataSource(df_proportions)

In [107]:
MAX_PROCEDURES = 8


top_procedures = (
    df_plot_aggregated.groupby('PROCEDURE_GROUP')['Avg_Wait_Time']
    .sum()
    .nlargest(MAX_PROCEDURES)
    .index.tolist()
)

df_plot_aggregated['PROCEDURE_GROUP_CLEAN'] = np.where(
    df_plot_aggregated['PROCEDURE_GROUP'].isin(top_procedures), 
    df_plot_aggregated['PROCEDURE_GROUP'], 
    'Other'
)

procedure_types = df_plot_aggregated['PROCEDURE_GROUP_CLEAN'].unique().tolist()

# Match palette length with your procedure count
palette = Category10[len(procedure_types)]

p_proc_top_cities = figure(
    y_range=city_order,
    height=600,
    width=900,
    title="Proportional Breakdown of Avg Wait Time by Procedure Group (Top 5 Cities)",
    x_axis_label="Proportion of Avg Wait Time",
    y_axis_label="City",
    tools="pan,wheel_zoom,box_zoom,reset,save,hover"
)

p_proc_top_cities.hbar_stack(
    stackers=procedure_types,
    y='City', 
    source=source_proc_top_cities, 
    height=0.6,
    color=palette,
    legend_label=procedure_types
)

p_proc_top_cities.xaxis.formatter = NumeralTickFormatter(format="0%")
p_proc_top_cities.x_range.start = 0

hover = HoverTool(tooltips=[
    ("City", "@City"),
    ("Total Avg Wait", "@Total_Wait_Time{0.1f} Days"),
    ("Proportion by Procedure", "@$name{0.0%}")
])
p_proc_top_cities.add_tools(hover)

p_proc_top_cities.legend.location = "top_right"
p_proc_top_cities.legend.orientation = "vertical"
p_proc_top_cities.ygrid.grid_line_color = None

show(p_proc_top_cities)
